# 01. Momentum & Funding Factor Construction
This notebook demonstrates how to load the clean panel and construct basic factors.

In [ ]:
import sys
sys.path.append('..')
from loader import load_panel
import polars as pl

# Load data lazy
lf = load_panel(start_date="2022-01-01", end_date="2022-02-01")

# Construct factors
# 1. Momentum (24h return)
# 2. Funding Factor (next funding rate)
factors = (
    lf
    .sort("timestamp")
    .with_columns([
        pl.col("close").pct_change(24).over("symbol").alias("mom_24h"),
        pl.col("funding_rate").fill_null(0).rolling_mean(3).over("symbol").alias("avg_funding_8h")
    ])
    .filter(pl.col("is_active"))
    .collect()
)

print(factors.head())